In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
import faiss
from functools import reduce
from datetime import timedelta
import datasets
import torch
from tqdm import tqdm
import os
from datetime import datetime

NUM_PROC = 32
CACHE_DIR = "/home/jupyter/filestore/storage/"

In [2]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_events_20230501"

dataset = load_from_disk(DATA_PATH)

In [3]:
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [4]:
polars_ds = dataset.to_polars()

IOStream.flush timed out


In [7]:
null_count_df = polars_ds.null_count()
null_count_df

user_id,stime,session_id,sequence_id,event_id,item_id,product_id,name,price,c0_name,c1_name,c2_name,brand_name,item_condition_name,size_name,color,user_segment
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,3647,1133524,3373871,209,11004016,16809568,0


In [5]:
polars_ds = polars_ds.with_columns(
    pl.col("c1_name").fill_null("NO_INFO"),
    pl.col("c2_name").fill_null("NO_INFO"),
    pl.col("brand_name").fill_null("NO_INFO"),
    pl.col("item_condition_name").fill_null("NO_INFO"),
    pl.col("size_name").fill_null("NO_INFO"),
    pl.col("color").fill_null("NO_INFO"),
)

polars_ds.null_count()

user_id,stime,session_id,sequence_id,event_id,item_id,product_id,name,price,c0_name,c1_name,c2_name,brand_name,item_condition_name,size_name,color,user_segment
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
most_popular_c0_names = (
    polars_ds
    .groupby("c0_name")
    .agg(pl.count("user_id").alias("events_cnt"))
    .sort("events_cnt", descending=True)
    .select("c0_name")
    .head(50)["c0_name"]
    .to_list()
)

most_popular_c1_names = (
    polars_ds
    .groupby("c1_name")
    .agg(pl.count("user_id").alias("events_cnt"))
    .sort("events_cnt", descending=True)
    .select("c1_name")
    .head(50)["c1_name"]
    .to_list()
)

most_popular_c2_names = (
    polars_ds
    .groupby("c2_name")
    .agg(pl.count("user_id").alias("events_cnt"))
    .sort("events_cnt", descending=True)
    .select("c2_name")
    .head(50)["c2_name"]
    .to_list()
)

In [7]:
event_types = polars_ds.select("event_id").unique()["event_id"].to_list()

In [8]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")

In [2]:
train_dataset = pl.read_parquet(f"{CACHE_DIR}/datasets/train_ranking_dataset.parquet")

In [4]:
train_dataset.select("user_id").unique().shape

(57122, 1)

In [10]:
SCORING_DT = "2023-05-15"

def filter_aggregation(agg_colname, filter_by=None, agg="cnt", dt_colname="date", scoring_dt=None, day_wnd=7, suffix=None):
    base_expr = pl.col(agg_colname)
    
    if filter_by:
        filter_colname, filter_value = filter_by
        base_expr = base_expr.filter(pl.col(filter_colname) == filter_value)
        
    if dt_colname and scoring_dt:
        start_dt = pd.to_datetime(scoring_dt) + timedelta(days=-day_wnd)
        end_dt = pd.to_datetime(scoring_dt)

        base_expr = (
            base_expr
            .filter(pl.col(dt_colname) >= start_dt)
            .filter(pl.col(dt_colname) < end_dt)
        )

    if agg == "last" or agg == "first":
        base_expr = base_expr.sort_by("stime", descending=False)
        
    agg_map = {
        "cnt": base_expr.count,
        "distinct": base_expr.n_unique,
        "max": base_expr.max,
        "mean": base_expr.mean,
        "min": base_expr.min,
        "std": base_expr.std,
        "sum": base_expr.sum,
        "last": base_expr.last,
        "first": base_expr.first
    }
    
    if agg not in agg_map:
        raise ValueError(f"Unknown aggregation: {agg}")
        
    col_name = f"{agg_colname}_{agg}"
    
    col_name = f"{filter_colname}={filter_value}_{col_name}" if filter_by else col_name
    col_name = f"{col_name}_wnd={day_wnd}" if dt_colname and scoring_dt else col_name
    col_name = f"{col_name}_{suffix}" if suffix else col_name

    return (
        agg_map[agg]()
        .alias(col_name)
    )

In [11]:
train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

In [12]:
user_feats = (
    train_interactions
    .join(
        train_dataset
        .select("user_id"),
        on="user_id",
        how="inner"
    )
    .groupby("user_id")
    .agg(
        filter_aggregation("price", filter_by=("event_id", "buy_comp"), agg="sum"),
        filter_aggregation("c0_name", agg="distinct"),
        filter_aggregation("c1_name", agg="distinct"),
        filter_aggregation("c2_name", agg="distinct"),
        filter_aggregation("brand_name", agg="distinct"),
        filter_aggregation("product_id", agg="distinct"),
        *[filter_aggregation("brand_name", filter_by=("event_id", event_type), agg="last") for event_type in event_types],
        *[filter_aggregation("c0_name", filter_by=("event_id", event_type), agg="last") for event_type in event_types],
        *[filter_aggregation("c1_name", filter_by=("event_id", event_type), agg="last") for event_type in event_types],
        *[filter_aggregation("c2_name", filter_by=("event_id", event_type), agg="last") for event_type in event_types],
        *[filter_aggregation("product_id", filter_by=("event_id", event_type), agg="distinct") for event_type in event_types],
        *[filter_aggregation("brand_name", filter_by=("event_id", event_type), agg="distinct") for event_type in event_types],
        *[filter_aggregation("c0_name", filter_by=("event_id", event_type), agg="distinct") for event_type in event_types],
        *[filter_aggregation("c1_name", filter_by=("event_id", event_type), agg="distinct") for event_type in event_types],
        *[filter_aggregation("c2_name", filter_by=("event_id", event_type), agg="distinct") for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="mean") for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="max") for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="min") for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="std") for event_type in event_types],
        *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="cnt") for event_type in event_types],
        *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="distinct") for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="mean", scoring_dt=SCORING_DT) for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="max", scoring_dt=SCORING_DT) for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="min", scoring_dt=SCORING_DT) for event_type in event_types],
        *[filter_aggregation("price", filter_by=("event_id", event_type), agg="std", scoring_dt=SCORING_DT) for event_type in event_types],
        *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="cnt", scoring_dt=SCORING_DT) for event_type in event_types],
        *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="distinct", scoring_dt=SCORING_DT) for event_type in event_types]
    )
)

In [13]:
user_feats.shape

(57122, 133)

In [ ]:
item_properties = ["c0_name", "c1_name", "c2_name", "brand_name", "item_condition_name", "size_name", "color", "price"]

item_feats = (
    train_interactions
    .join(
        train_dataset
        .select("item_id"),
        on="item_id",
        how="inner"
    )
    .groupby("item_id")
    .agg(
        *[pl.first(col_name) for col_name in item_properties],
        *[filter_aggregation("user_id", filter_by=("event_id", event_type), agg="cnt") for event_type in ["item_add_to_cart_tap", "item_view", "item_like"]],
        *[filter_aggregation("user_id", filter_by=("event_id", event_type), agg="distinct") for event_type in ["item_add_to_cart_tap", "item_view", "item_like"]],
        *[filter_aggregation("user_id", filter_by=("event_id", event_type), agg="cnt", scoring_dt=SCORING_DT) for event_type in ["item_add_to_cart_tap", "item_view", "item_like"]],
        *[filter_aggregation("user_id", filter_by=("event_id", event_type), agg="distinct", scoring_dt=SCORING_DT) for event_type in ["item_add_to_cart_tap", "item_view", "item_like"]],
    )
)

Unknown instance spec: Please select VM configuration

In [15]:
user_feats.shape, item_feats.shape

((57122, 133), (304763, 21))

In [16]:
join_on = {}

for col_name in ["c0_name", "c1_name", "c2_name", "brand_name", "product_id", "item_condition_name", "size_name", "color"]:
    unique_col_values = (
        train_interactions
        .join(
            train_dataset
            .select("item_id"),
            on="item_id",
            how="inner"
        )
        .select(col_name)
        .unique()
    
    )
    
    feats = (
        train_interactions
        .join(
            train_dataset
            .select("user_id"),
            on="user_id",
            how="inner"
        )
        .join(
            unique_col_values,
            on=col_name,
            how="inner",
        )
        .groupby("user_id", col_name)
        .agg(
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="mean", suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="max", suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="min", suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="std", suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="cnt", suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="distinct", suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="mean", scoring_dt=SCORING_DT, suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="max", scoring_dt=SCORING_DT, suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="min", scoring_dt=SCORING_DT, suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("price", filter_by=("event_id", event_type), agg="std", scoring_dt=SCORING_DT, suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="cnt", scoring_dt=SCORING_DT, suffix=f"by={col_name}") for event_type in event_types],
            *[filter_aggregation("item_id", filter_by=("event_id", event_type), agg="distinct", scoring_dt=SCORING_DT, suffix=f"by={col_name}") for event_type in event_types]
        )
    )
    
    join_on[col_name] = feats

In [18]:
dataset_with_feats = (
    train_dataset
    .join(
        user_feats,
        on="user_id",
        how="left"
    )
    .join(
        item_feats,
        on="item_id",
        how="left",
    )
)

for col_name in join_on:
    if col_name != "product_id":
        dataset_with_feats = dataset_with_feats.join(join_on[col_name], on=[col_name, "user_id"], how="left")

In [19]:
dataset_with_feats.shape

(1425658, 659)

In [20]:
dataset_with_feats.write_parquet("/home/jupyter/filestore/storage/datasets/train_dataset_with_feats.parquet")